# blueprint-vlm on Kaggle (2x T4)

Settings -> **Accelerator: GPU T4 x2**, **Internet: On**.

T4 is Turing and has no bfloat16, so this uses fp16 throughout. The two cards
are used as two DDP ranks with a full model copy each, not as one 32GB pool --
they are separate 16GB cards and a 3B model in 4-bit fits on one comfortably.

In [ ]:
# Preflight. Run this before anything else -- it is cheap and it catches the
# two failures that waste a whole session: no accelerator, and no internet.
import shutil, torch, socket

print('nvidia-smi :', shutil.which('nvidia-smi') or 'NOT FOUND (session is CPU-only)')
print('cuda       :', torch.cuda.is_available(), '| devices:', torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        d = torch.cuda.get_device_properties(i)
        print(f'  [{i}] {d.name}  {d.total_memory/1e9:.0f} GB  sm_{d.major}{d.minor}')
    print('bf16       :', torch.cuda.is_bf16_supported(), '(False on T4 -> fp16 path, expected)')
else:
    raise SystemExit(
        'No GPU attached.\n'
        '  Sidebar -> Session options -> Accelerator -> "GPU T4 x2", then re-run from the top.\n'
        '  Changing the accelerator restarts the session.\n'
        '  Greyed out? Kaggle needs a verified phone number before it grants GPUs.\n'
        '  Still nothing? Check the 30 hr/week GPU quota (resets Saturday).'
    )

try:
    socket.create_connection(('huggingface.co', 443), timeout=5)
    print('internet   : on')
except OSError:
    raise SystemExit('No internet -- sidebar -> Session options -> Internet: On. '
                     'Needed to pull the 7GB model from Hugging Face.')


In [ ]:
# Fetch the repo. `git clone` FAILS if the directory already exists, and
# /kaggle/working survives between sessions -- so a re-run would silently keep
# running whatever code was checked out days ago. Reset to origin/main instead.
import os, subprocess
os.environ['HF_HOME'] = '/kaggle/working/hf'
os.environ['PYTHONPATH'] = 'src'
REPO = '/kaggle/working/blueprint-vlm'
URL  = 'https://github.com/baohq1595/blueprint-vlm.git'

if os.path.isdir(REPO + '/.git'):
    subprocess.run(['git', '-C', REPO, 'fetch', '--all', '-q'], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', URL, REPO], check=True)

%cd {REPO}
!pip install -q -U transformers peft accelerate bitsandbytes qwen-vl-utils
# Kaggle preinstalls torchao 0.10.0. The peft we just upgraded requires >0.16.0,
# and peft's is_torchao_available() RAISES ImportError on an old version instead
# of returning False -- so every LoRA module dispatch dies at load time.
# We never use torchao (4-bit comes from bitsandbytes), so remove it: with the
# package absent the same function returns False cleanly and PEFT moves on.
!pip uninstall -y -q torchao || true
!python -c "import importlib.util as u; print('torchao present:', u.find_spec('torchao') is not None)"

# Print exactly which commit is about to run. If this does not match the newest
# commit on GitHub, nothing below is testing the code you think it is.
!git log --oneline -1
!git status -sb | head -1

In [ ]:
!python scripts/build_dataset.py --out data/synth

## 1. Zero-shot baseline
Single GPU is fine for inference. ~20 min for the 1600-record test split.

In [ ]:
# Zero-shot baseline across BOTH cards: one process per GPU, contiguous shards.
#
# CUDA_VISIBLE_DEVICES isolates each process to one card, so neither can
# allocate on the other. This is data parallelism -- two full model copies --
# which is the right shape here. device_map="auto" would instead split ONE model
# across both cards and leave the second idle at 0% util.
#
# --limit 400 keeps the baseline to roughly 15 min. Read the rec/s the run
# prints, then decide whether to drop the limit.
import os, subprocess

def run_shards(out_prefix, adapter=None, split="data/synth/test", limit=400):
    procs = []
    for i in range(2):
        cmd = ["python", "-m", "blueprint_vlm.infer",
               "--config", "configs/qwen25vl_3b_lora_quick.yaml",
               "--split", split, "--out", f"{out_prefix}_s{i}.json",
               "--device", "0", "--num-shards", "2", "--shard-id", str(i),
               "--batch-size", "8"]
        if limit:   cmd += ["--limit", str(limit)]
        if adapter: cmd += ["--adapter", adapter]
        env = dict(os.environ, PYTHONPATH="src", CUDA_VISIBLE_DEVICES=str(i))
        procs.append(subprocess.Popen(cmd, env=env))
    for pr in procs:
        assert pr.wait() == 0, "a shard failed -- scroll up for its traceback"
    subprocess.run(["python", "scripts/merge_shards.py",
                    "--shards", f"{out_prefix}_s0.json", f"{out_prefix}_s1.json",
                    "--out", f"{out_prefix}.json"], check=True)

run_shards("results/zeroshot_test")


In [ ]:
!PYTHONPATH=src python scripts/score.py --split data/synth/test \
    --predictions results/zeroshot_test.json \
    --out results/zeroshot_test_scored.json --title 'zero-shot / test (n=400)'


## 2. Fine-tune across both cards
4000 records x 2 epochs. Budget 3-5 hours; the session limit is 12.

In [ ]:
!PYTHONPATH=src accelerate launch --multi_gpu --num_processes 2 \
    --mixed_precision fp16 --module blueprint_vlm.train \
    --config configs/qwen25vl_3b_lora_quick.yaml

## 3. Score both held-out sets
`test_hard` is the shifted distribution. Report it even when it is bad.

In [ ]:
# Fine-tuned, same sharding. Drop limit=None once you know the rate.
run_shards("results/finetuned_test", adapter="/kaggle/working/outputs/qwen25vl-3b-quick")
run_shards("results/finetuned_hard", adapter="/kaggle/working/outputs/qwen25vl-3b-quick",
           split="data/synth/test_hard", limit=200)


In [ ]:
!PYTHONPATH=src python scripts/score.py --split data/synth/test \
    --predictions results/finetuned_test.json \
    --out results/finetuned_test_scored.json --title 'fine-tuned / test (n=400)'
!PYTHONPATH=src python scripts/score.py --split data/synth/test_hard \
    --predictions results/finetuned_hard.json \
    --out results/finetuned_hard_scored.json --title 'fine-tuned / test_hard (n=200)'


## 4. If you are short on time
Run the baseline and the fine-tune on `test` only. `test_hard` and the
resolution ablation can go in a second session -- the adapter is saved to
`/kaggle/working` and can be added as a dataset to the next notebook.